### Step 1 – Install Dependencies & Load GPT-2

In [1]:
# Install all dependencies first, then import in the same cell
# so the kernel picks them up without needing a restart.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'transformers', 'datasets', 'accelerate', '-q'])

import torch
import math
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import Dataset

set_seed(42)

# Load pre-trained GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# GPT-2 has no pad token by default; reuse the EOS token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print('GPT-2 model and tokenizer loaded successfully.')
print(f'Number of parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device available: {"GPU (CUDA)" if torch.cuda.is_available() else "CPU"}')

C:\Users\Apoorv Bagga\AppData\Roaming\Python\Python310\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\Users\Apoorv Bagga\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

C:\Users\Apoorv Bagga\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Apoorv Bagga\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 model and tokenizer loaded successfully.
Number of parameters: 124,439,808
Device available: GPU (CUDA)


### Step 2 – Generate Baseline Reviews (Before Fine-Tuning)

We first capture what GPT-2 generates **before** any fine-tuning. Expect generic, non-review-style output.

In [2]:
def generate_text(model, tokenizer, prompt, max_length=100):
    """Generate text from a prompt using the given model and tokenizer."""
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


review_prompts = [
    'This product is',
    'I bought this phone and',
    'The quality of this item',
]

print('=== BASELINE REVIEWS (Before Fine-Tuning) ===')
baseline = {}
for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f'Prompt : {p}')
    print(f'Output : {baseline[p]}')
    print()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE REVIEWS (Before Fine-Tuning) ===
Prompt : This product is
Output : This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most

Prompt : I bought this phone and
Output : I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 pro

### Step 3 – Prepare Dataset & Fine-Tune

We create a small domain-specific corpus of 20 realistic product reviews, tokenize it, and run 15 epochs of causal-language-modeling training.

In [3]:
# ── Product Review Corpus ──────────────────────────────────────────────────
corpus = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

# Build a Hugging Face Dataset and tokenize
dataset = Dataset.from_dict({'text': corpus})
tokenized = dataset.map(
    lambda x: tokenizer(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)
split = tokenized.train_test_split(test_size=0.15, seed=42)

print(f'Training samples : {len(split["train"])}')
print(f'Evaluation samples: {len(split["test"])}')

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Training samples : 17
Evaluation samples: 3


In [4]:
# ── Training Configuration ────────────────────────────────────────────────
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./gpt2-reviews',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),   # Use FP16 if a GPU is available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator,
)

print('Starting fine-tuning for Product Review Generator...')
trainer.train()
print('Fine-tuning complete!')

Starting fine-tuning for Product Review Generator...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.350773
2,3.970800,3.246724
3,3.970800,3.104579
4,3.341200,2.981637
5,3.341200,2.881284
6,2.755200,2.804400
7,2.755200,2.762120
8,1.905400,2.748276
9,1.905400,2.762690
10,1.276400,2.876994


Fine-tuning complete!


### Step 4 – Evaluate & Compare Outputs

In [5]:
# Perplexity (lower is better; measures how well the model predicts the eval set)
eval_res = trainer.evaluate()
perplexity = math.exp(eval_res['eval_loss'])
print(f'Evaluation Loss : {eval_res["eval_loss"]:.4f}')
print(f'Perplexity      : {perplexity:.2f}')
print()

# Side-by-side comparison
print('=== FINE-TUNED REVIEWS (After Fine-Tuning) ===')
for p in review_prompts:
    ft_out = generate_text(model, tokenizer, p)
    print(f'Prompt      : {p}')
    print(f'  Baseline  : {baseline[p][:120]}')
    print(f'  Fine-Tuned: {ft_out[:120]}')
    print()

Evaluation Loss : 3.4279
Perplexity      : 30.81

=== FINE-TUNED REVIEWS (After Fine-Tuning) ===
Prompt      : This product is
  Baseline  : This product is made from high quality, lightweight stainless steel. If you are looking for something a little more dura
  Fine-Tuned: This product is packed with features that make this purchase even more worthwhile. Please take a moment to review these 

Prompt      : I bought this phone and
  Baseline  : I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen 
  Fine-Tuned: I bought this phone and i would definitely buy from this brand again.

Prompt      : The quality of this item
  Baseline  : The quality of this item in the item description (and if the item is already in stock) will determine how many times the
  Fine-Tuned: The quality of this item is outstanding value for money and the build quality feels premium despite the affordable price



### Step 1 – Reload a Fresh GPT-2 Model

In [6]:
# Important: load a fresh (unfine-tuned) copy of GPT-2 for this component
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

print('Fresh GPT-2 model loaded for Component II.')

Fresh GPT-2 model loaded for Component II.


### Step 2 – Generate Baseline Recipes (Before Fine-Tuning)

In [7]:
recipe_prompts = [
    'To make butter chicken',
    'For pasta carbonara',
    'To prepare a chocolate cake',
]

print('=== BASELINE RECIPES (Before Fine-Tuning) ===')
baseline2 = {}
for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f'Prompt : {p}')
    print(f'Output : {baseline2[p]}')
    print()

=== BASELINE RECIPES (Before Fine-Tuning) ===
Prompt : To make butter chicken
Output : To make butter chicken with eggs. I was going to do this with a different recipe, but I was going into a different area of town and was going to have to do something with them. I love a good egg.

Here's how to make your own:

1 pound unsalted butter

1 tsp vanilla extract

8-10 eggs

1 cup all-purpose flour

4 tablespoons unsalted butter

1/2 cup all-purpose flour


Prompt : For pasta carbonara
Output : For pasta carbonara pasta

2 large olives, cut into thin strips

1 large onion, finely chopped

4 small garlic cloves, finely chopped

3 large red bell peppers, finely chopped

1 large green chile, finely chopped

4 cups water

1 tsp dried oregano or ground oregano

salt to taste

4 cups white wine vinegar

1 tbsp olive oil

1 tsp salt

1 cup water

Prompt : To prepare a chocolate cake
Output : To prepare a chocolate cake, you're going to want a cake that is pretty light. You can also use a light bro

### Step 3 – Prepare Recipe Dataset & Fine-Tune

In [8]:
# ── Recipe Instruction Corpus ─────────────────────────────────────────────
recipes = [
    'to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.',
    'heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.',
    'add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.',
    'add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.',
    'finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.',
    'for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.',
    'fry diced pancetta in olive oil until crispy and set aside.',
    'whisk together egg yolks parmesan cheese and black pepper in a bowl.',
    'toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.',
    'the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.',
    'to prepare vegetable stir fry heat sesame oil in a wok on high heat.',
    'add sliced bell peppers broccoli florets and snap peas and toss for three minutes.',
    'pour in soy sauce oyster sauce and a pinch of sugar and stir well.',
    'add minced garlic and ginger and cook for one more minute until fragrant.',
    'serve the stir fry over steamed jasmine rice and garnish with sesame seeds.',
    'for chocolate chip cookies cream together butter and sugar until light and fluffy.',
    'beat in eggs one at a time then add vanilla extract and mix well.',
    'fold in flour baking soda and salt then gently stir in chocolate chips.',
    'scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.',
    'let cookies cool on the tray for five minutes before transferring to a wire rack.',
]

dataset2 = Dataset.from_dict({'text': recipes})
tok2 = dataset2.map(
    lambda x: tokenizer2(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)
split2 = tok2.train_test_split(test_size=0.15, seed=42)

print(f'Training samples : {len(split2["train"])}')
print(f'Evaluation samples: {len(split2["test"])}')

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Training samples : 17
Evaluation samples: 3


In [9]:
# ── Training Configuration ────────────────────────────────────────────────
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2['train'],
    eval_dataset=split2['test'],
    data_collator=collator2,
)

print('Starting fine-tuning for Recipe Instruction Generator...')
trainer2.train()
print('Fine-tuning complete!')

Starting fine-tuning for Recipe Instruction Generator...


Epoch,Training Loss,Validation Loss
1,No log,3.419763
2,3.893000,3.260991
3,3.893000,3.099674
4,3.427700,2.971442
5,3.427700,2.917478
6,2.707000,2.892458
7,2.707000,2.870103
8,2.169700,2.878086
9,2.169700,3.000883
10,1.642800,3.125551


Fine-tuning complete!


### Step 4 – Evaluate & Compare Outputs

In [10]:
eval2 = trainer2.evaluate()
perplexity2 = math.exp(eval2['eval_loss'])
print(f'Evaluation Loss : {eval2["eval_loss"]:.4f}')
print(f'Perplexity      : {perplexity2:.2f}')
print()

print('=== FINE-TUNED RECIPES (After Fine-Tuning) ===')
for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)
    print(f'Prompt      : {p}')
    print(f'  Baseline  : {baseline2[p][:120]}')
    print(f'  Fine-Tuned: {ft[:120]}')
    print()

Evaluation Loss : 3.5258
Perplexity      : 33.98

=== FINE-TUNED RECIPES (After Fine-Tuning) ===
Prompt      : To make butter chicken
  Baseline  : To make butter chicken with eggs. I was going to do this with a different recipe, but I was going into a different area 
  Fine-Tuned: To make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one 

Prompt      : For pasta carbonara
  Baseline  : For pasta carbonara pasta

2 large olives, cut into thin strips

1 large onion, finely chopped

4 small garlic cloves, f
  Fine-Tuned: For pasta carbonara boil pasta in salted water until al dente and reserve half cup of pasta water. Add pasta carbonara a

Prompt      : To prepare a chocolate cake
  Baseline  : To prepare a chocolate cake, you're going to want a cake that is pretty light. You can also use a light brown cake or a 
  Fine-Tuned: To prepare a chocolate cake dough press place dough onto a baking sheet and bake at 180 degrees for